### loading

In [1]:
import os

In [2]:
# Define a User-Agent string for macOS
os.environ["USER_AGENT"] = "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/114.0.0.0 Safari/537.36"

In [3]:
from langchain_community.document_loaders import WebBaseLoader

In [4]:
#삼엽충의 고도로 복잡한 눈! 
url = "https://creation.kr/Circulation/?idx=1295059&bmode=view"

### Crawling

In [5]:
import bs4

In [6]:
loader = WebBaseLoader(
    web_path = url,
    verify_ssl = False,
    bs_kwargs=dict(
        parse_only=bs4.SoupStrainer(
            class_=("margin-top-xxl _comment_body_")
        )
    ),
)

In [7]:
page = loader.load()

/opt/miniconda3/envs/llm/lib/python3.9/site-packages/urllib3/connectionpool.py:1099: InsecureRequestWarning: Unverified HTTPS request is being made to host 'creation.kr'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


In [8]:
page

[Document(metadata={'source': 'https://creation.kr/Circulation/?idx=1295059&bmode=view'}, page_content='삼엽충의 고도로 복잡한 눈!\xa0(Trilobites — The Eyes Have It!)Frank Sherwin, Mark Armitage\xa0 \xa0 \xa0 삼엽충(trilobites)으로 알려진 멸종된 절지동물은 지질주상도(geologic column) 상의 고생대 지층에서 흔하게 발견되고 있다. 최초 삼엽충은 진화론적 시간 틀로 5억2천만 년 전인, 초기 캄브리아기의 상부 지층에서 발견된다. 그리고 그들은 페름기 (2억 년 전으로 추정)까지 확장되어 발견되고 있다.그림 1. 아리조나주 홀 브룩에 있는 화석가게에서 구해진 미확인된, 대략 5cm 정도 길이의 삼엽충 화석. 삼엽충 눈에 대한 광학현미경 사진은 이들이 고도로 발달된 눈을 가지고 있음을 보여준다. 스키조크로알 눈(Schizochroal eyes)의 렌즈 뭉치(lens assembly, 흰색 화살표가 가리키는 커다란 튀어나온 부분들)를 볼 수 있는데, 일부는 떨어져나가 부서져있다. (사진을 위해 금속 막대 위에 올려졌다). 렌즈 뭉치의 아크(왼쪽에서 오른쪽으로 검은 화살표 사이)는 180+ O이다. 스케일 바는 400㎛ 이다.모든 절지동물들처럼, 삼엽충은 쌍으로 된 관절 부속지 및 키틴질의 외골격을 가지고 있다. 오소리오(Osorio et al. 1997, p 244) 등이 언급했던 것처럼, 일반적으로 절지동물의 기원은, 특히 삼엽충의 기원은 진화론자들에게 하나의 미스터리이다.다윈이 ‘종의 기원(Origin of Species)’에서도 언급했던 바와 같이, 화석기록에서 캄브리아기 동안에 절지동물이 갑작스럽게 출현하는 것은 진화 생물학에 하나의 심각한 문제이다. 현대의 절지동물이 벌레 같은 조상에서 어떻게 진화했는지를 보여주는, 명백히 단순하게 생긴 중간형태의 전이생물은 화석기록이나, 살아있는 생물 종에서 관

### Splitting

In [9]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [10]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 500,
    chunk_overlap=100
)

In [11]:
docs = text_splitter.split_documents(page)

### Embedding

In [12]:
from langchain_openai import OpenAIEmbeddings

In [13]:
embedding_model = OpenAIEmbeddings(model="text-embedding-3-large")

In [14]:
text_content = [text.page_content for text in docs]

In [15]:
embeddings = embedding_model.embed_documents(text_content)

In [16]:
len(embeddings)

16

In [17]:
from langchain.vectorstores import Chroma

In [20]:
db = Chroma.from_documents(docs, embedding_model)

### similarity search

In [21]:
query = "삼엽충 화석은 진화론을 뒷받침하고 있나요?"

In [22]:
db.similarity_search(query)

[Document(metadata={'source': 'https://creation.kr/Circulation/?idx=1295059&bmode=view'}, page_content='바는 120㎛.삼엽충 연구를 하고 있는 많은 고생물학자들은, 진화론은 점진적 과정이라고 믿고 있다. 그들은 한 형태의 삼엽충이 다른 형태의 삼엽충으로 변화되어 가는 것을 볼 수 있을 것으로 생각하고 있다. 그러나 단속평형(punctuated equilibrium) 모델을 믿고 있는 사람들은, 삼엽충은 상당 기간 동안 어떠한 변화도 보이지 않다가 빠르게 폭발적으로 진화적 혁신을 일으켰을 것으로 믿고 있다.(Eldredge 1985; Eldredge and Gould 1972). 클락슨과 다른 사람들은 삼엽충 멸종을 일으켰던 많은 사건들이 발생했고, 삼엽충을 다시 원점으로 돌아가게 했다는 이론을 제시하며 중도적 입장을 가지고 있다.(Clarkson 1986; Palmer 1999). 그러나 삼엽충의 발전을 위해 제안되는 진화 메커니즘이 무엇이든지 간에, 삼엽충 분류 분야는 혼란에 빠져있다. 한 연구자는(진화론적으로 생물 다양성을 설명하려는) ”계통분류학(systematics)은 아직도'),
 Document(metadata={'source': 'https://creation.kr/Circulation/?idx=1295059&bmode=view'}, page_content='등을 알고 있었음이 분명하다.”(Levi-Setti, 1993, p 33). 물론 이러한 말은 명백히 불합리하다. 왜냐하면, 절지동물은 광학의 법칙들을 알고 있을 수 없기 때문이다. 따라서 진화가 이러한 놀라운 생물학적 렌즈들을 설명할 수 없다는 것은 분명하다.그림 3. 미확인된 삼엽충 표본의 스키조크로알 렌즈 뭉치에 대한 주사전자현미경 사진. 표본은 주사전자현미경(SEM) 스퍼터 코팅기 위에서 팔라듐으로 2분 동안 코팅되었고, 관찰되었고, JEOL 35 SEM 위에서 촬영되었다. 융기되어있는 각 부분은 렌즈이다

In [23]:
# Search with Score
db.similarity_search_with_score(query)

[(Document(metadata={'source': 'https://creation.kr/Circulation/?idx=1295059&bmode=view'}, page_content='바는 120㎛.삼엽충 연구를 하고 있는 많은 고생물학자들은, 진화론은 점진적 과정이라고 믿고 있다. 그들은 한 형태의 삼엽충이 다른 형태의 삼엽충으로 변화되어 가는 것을 볼 수 있을 것으로 생각하고 있다. 그러나 단속평형(punctuated equilibrium) 모델을 믿고 있는 사람들은, 삼엽충은 상당 기간 동안 어떠한 변화도 보이지 않다가 빠르게 폭발적으로 진화적 혁신을 일으켰을 것으로 믿고 있다.(Eldredge 1985; Eldredge and Gould 1972). 클락슨과 다른 사람들은 삼엽충 멸종을 일으켰던 많은 사건들이 발생했고, 삼엽충을 다시 원점으로 돌아가게 했다는 이론을 제시하며 중도적 입장을 가지고 있다.(Clarkson 1986; Palmer 1999). 그러나 삼엽충의 발전을 위해 제안되는 진화 메커니즘이 무엇이든지 간에, 삼엽충 분류 분야는 혼란에 빠져있다. 한 연구자는(진화론적으로 생물 다양성을 설명하려는) ”계통분류학(systematics)은 아직도'),
  0.9339120388031006),
 (Document(metadata={'source': 'https://creation.kr/Circulation/?idx=1295059&bmode=view'}, page_content='등을 알고 있었음이 분명하다.”(Levi-Setti, 1993, p 33). 물론 이러한 말은 명백히 불합리하다. 왜냐하면, 절지동물은 광학의 법칙들을 알고 있을 수 없기 때문이다. 따라서 진화가 이러한 놀라운 생물학적 렌즈들을 설명할 수 없다는 것은 분명하다.그림 3. 미확인된 삼엽충 표본의 스키조크로알 렌즈 뭉치에 대한 주사전자현미경 사진. 표본은 주사전자현미경(SEM) 스퍼터 코팅기 위에서 팔라듐으로 2분 동안 코팅되었고, 관찰되었고, JEOL 35 SEM 위에서

In [25]:
# return the score in range [0,1]
# 0 = 아무런 상관 없음; 1 = 아주 상관 있음
db.similarity_search_with_relevance_scores(query)

[(Document(metadata={'source': 'https://creation.kr/Circulation/?idx=1295059&bmode=view'}, page_content='바는 120㎛.삼엽충 연구를 하고 있는 많은 고생물학자들은, 진화론은 점진적 과정이라고 믿고 있다. 그들은 한 형태의 삼엽충이 다른 형태의 삼엽충으로 변화되어 가는 것을 볼 수 있을 것으로 생각하고 있다. 그러나 단속평형(punctuated equilibrium) 모델을 믿고 있는 사람들은, 삼엽충은 상당 기간 동안 어떠한 변화도 보이지 않다가 빠르게 폭발적으로 진화적 혁신을 일으켰을 것으로 믿고 있다.(Eldredge 1985; Eldredge and Gould 1972). 클락슨과 다른 사람들은 삼엽충 멸종을 일으켰던 많은 사건들이 발생했고, 삼엽충을 다시 원점으로 돌아가게 했다는 이론을 제시하며 중도적 입장을 가지고 있다.(Clarkson 1986; Palmer 1999). 그러나 삼엽충의 발전을 위해 제안되는 진화 메커니즘이 무엇이든지 간에, 삼엽충 분류 분야는 혼란에 빠져있다. 한 연구자는(진화론적으로 생물 다양성을 설명하려는) ”계통분류학(systematics)은 아직도'),
  0.33857859028477566),
 (Document(metadata={'source': 'https://creation.kr/Circulation/?idx=1295059&bmode=view'}, page_content='등을 알고 있었음이 분명하다.”(Levi-Setti, 1993, p 33). 물론 이러한 말은 명백히 불합리하다. 왜냐하면, 절지동물은 광학의 법칙들을 알고 있을 수 없기 때문이다. 따라서 진화가 이러한 놀라운 생물학적 렌즈들을 설명할 수 없다는 것은 분명하다.그림 3. 미확인된 삼엽충 표본의 스키조크로알 렌즈 뭉치에 대한 주사전자현미경 사진. 표본은 주사전자현미경(SEM) 스퍼터 코팅기 위에서 팔라듐으로 2분 동안 코팅되었고, 관찰되었고, JEOL 35 SEM 위에

## Retriever

In [26]:
retriever = db.as_retriever()

In [28]:
out_docs = retriever.invoke(query)

for docs in out_docs:
    print(docs.page_content)
    print("="*60)

바는 120㎛.삼엽충 연구를 하고 있는 많은 고생물학자들은, 진화론은 점진적 과정이라고 믿고 있다. 그들은 한 형태의 삼엽충이 다른 형태의 삼엽충으로 변화되어 가는 것을 볼 수 있을 것으로 생각하고 있다. 그러나 단속평형(punctuated equilibrium) 모델을 믿고 있는 사람들은, 삼엽충은 상당 기간 동안 어떠한 변화도 보이지 않다가 빠르게 폭발적으로 진화적 혁신을 일으켰을 것으로 믿고 있다.(Eldredge 1985; Eldredge and Gould 1972). 클락슨과 다른 사람들은 삼엽충 멸종을 일으켰던 많은 사건들이 발생했고, 삼엽충을 다시 원점으로 돌아가게 했다는 이론을 제시하며 중도적 입장을 가지고 있다.(Clarkson 1986; Palmer 1999). 그러나 삼엽충의 발전을 위해 제안되는 진화 메커니즘이 무엇이든지 간에, 삼엽충 분류 분야는 혼란에 빠져있다. 한 연구자는(진화론적으로 생물 다양성을 설명하려는) ”계통분류학(systematics)은 아직도
등을 알고 있었음이 분명하다.”(Levi-Setti, 1993, p 33). 물론 이러한 말은 명백히 불합리하다. 왜냐하면, 절지동물은 광학의 법칙들을 알고 있을 수 없기 때문이다. 따라서 진화가 이러한 놀라운 생물학적 렌즈들을 설명할 수 없다는 것은 분명하다.그림 3. 미확인된 삼엽충 표본의 스키조크로알 렌즈 뭉치에 대한 주사전자현미경 사진. 표본은 주사전자현미경(SEM) 스퍼터 코팅기 위에서 팔라듐으로 2분 동안 코팅되었고, 관찰되었고, JEOL 35 SEM 위에서 촬영되었다. 융기되어있는 각 부분은 렌즈이다. 작게 융기된 부분들은 미확인된 구조로, 아직 그 기능은 알려지지 않고 있다. 스케일 바는 150㎛. 그림 4. 미확인된 삼엽충 표본의 스키조크로알 렌즈 뭉치에 대한 주사전자현미경 사진. 스케일 바는 120㎛.삼엽충 연구를 하고 있는 많은 고생물학자들은, 진화론은 점진적 과정이라고 믿고 있다. 그들은 한 형태의 삼엽충이 다른 형태의 삼엽충으로 변화되어 가는 것을 볼 수
제안

### Most Relevant

In [29]:
retriever = db.as_retriever(search_kwargs={"k":1})

In [30]:
out_docs = retriever.invoke(query)
print(out_docs[0].page_content)

바는 120㎛.삼엽충 연구를 하고 있는 많은 고생물학자들은, 진화론은 점진적 과정이라고 믿고 있다. 그들은 한 형태의 삼엽충이 다른 형태의 삼엽충으로 변화되어 가는 것을 볼 수 있을 것으로 생각하고 있다. 그러나 단속평형(punctuated equilibrium) 모델을 믿고 있는 사람들은, 삼엽충은 상당 기간 동안 어떠한 변화도 보이지 않다가 빠르게 폭발적으로 진화적 혁신을 일으켰을 것으로 믿고 있다.(Eldredge 1985; Eldredge and Gould 1972). 클락슨과 다른 사람들은 삼엽충 멸종을 일으켰던 많은 사건들이 발생했고, 삼엽충을 다시 원점으로 돌아가게 했다는 이론을 제시하며 중도적 입장을 가지고 있다.(Clarkson 1986; Palmer 1999). 그러나 삼엽충의 발전을 위해 제안되는 진화 메커니즘이 무엇이든지 간에, 삼엽충 분류 분야는 혼란에 빠져있다. 한 연구자는(진화론적으로 생물 다양성을 설명하려는) ”계통분류학(systematics)은 아직도


### Maximal Marginal Relevance(MMR)
* recommendation 시스템에서, 사용자가 좋아하는 주제에 대해 다양한 장르/양식을 추천해주는 것
* `lambda_mult`
    * 0.5 = neutral
    * 0.0 = diversity
    * 1.0 = similarity

In [32]:
retriever = db.as_retriever(
    search_type = "mmr",
    search_kwargs = {"k": 3, "fetch_k": 5, "lambda_mult": 0.8}
)

In [34]:
out_docs = retriever.invoke(query)

for doc in out_docs:
    print(doc.page_content)
    print("="*60)

바는 120㎛.삼엽충 연구를 하고 있는 많은 고생물학자들은, 진화론은 점진적 과정이라고 믿고 있다. 그들은 한 형태의 삼엽충이 다른 형태의 삼엽충으로 변화되어 가는 것을 볼 수 있을 것으로 생각하고 있다. 그러나 단속평형(punctuated equilibrium) 모델을 믿고 있는 사람들은, 삼엽충은 상당 기간 동안 어떠한 변화도 보이지 않다가 빠르게 폭발적으로 진화적 혁신을 일으켰을 것으로 믿고 있다.(Eldredge 1985; Eldredge and Gould 1972). 클락슨과 다른 사람들은 삼엽충 멸종을 일으켰던 많은 사건들이 발생했고, 삼엽충을 다시 원점으로 돌아가게 했다는 이론을 제시하며 중도적 입장을 가지고 있다.(Clarkson 1986; Palmer 1999). 그러나 삼엽충의 발전을 위해 제안되는 진화 메커니즘이 무엇이든지 간에, 삼엽충 분류 분야는 혼란에 빠져있다. 한 연구자는(진화론적으로 생물 다양성을 설명하려는) ”계통분류학(systematics)은 아직도
등을 알고 있었음이 분명하다.”(Levi-Setti, 1993, p 33). 물론 이러한 말은 명백히 불합리하다. 왜냐하면, 절지동물은 광학의 법칙들을 알고 있을 수 없기 때문이다. 따라서 진화가 이러한 놀라운 생물학적 렌즈들을 설명할 수 없다는 것은 분명하다.그림 3. 미확인된 삼엽충 표본의 스키조크로알 렌즈 뭉치에 대한 주사전자현미경 사진. 표본은 주사전자현미경(SEM) 스퍼터 코팅기 위에서 팔라듐으로 2분 동안 코팅되었고, 관찰되었고, JEOL 35 SEM 위에서 촬영되었다. 융기되어있는 각 부분은 렌즈이다. 작게 융기된 부분들은 미확인된 구조로, 아직 그 기능은 알려지지 않고 있다. 스케일 바는 150㎛. 그림 4. 미확인된 삼엽충 표본의 스키조크로알 렌즈 뭉치에 대한 주사전자현미경 사진. 스케일 바는 120㎛.삼엽충 연구를 하고 있는 많은 고생물학자들은, 진화론은 점진적 과정이라고 믿고 있다. 그들은 한 형태의 삼엽충이 다른 형태의 삼엽충으로 변화되어 가는 것을 볼 수
삼엽